# Palace Eigenmode and Surface EPR for a QPDK Transmon

This notebook builds a public QPDK double-pad transmon, compiles its authored
component and process semantics with SCGSim, and specifies a manual single-node
Slurm handoff for a two-mode Palace eigenmode solve. It never submits or runs the
solver.

**Current limitation:** the pinned SCGSim revision rejects the requested
`metal_gap_equivalent` profile for this single-face M1 stack with
`ValueError: Route A requires exactly two physical face-metal Z ranges.`
Stack compilation is supported, but mesh preparation stops at that check; config,
handoff, solver execution, and returned-run analysis below are not validated.
The supported single-face profile, `substrate_face`, would also require compatible
vacuum-host bounds that contain M1. This notebook retains the requested profile
and physical stack; it does not add an artificial second metal face.

The installation command pins the source revision. SCGSim handoff receipts record
its package version, not the Git commit; a receipt alone does not prove that pin.

Use Python 3.12 and install the public, revision-pinned SCGSim package alongside
this QPDK checkout:

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install -e .
python -m pip install \
  "scgsim[palace,visualization] @ git+https://github.com/OrPenStrike/scgsim.git@de446a96c74f56ac06dbcd39776bee8372b74311"
```

Palace itself is an external executable. The handoff stage records the caller's
executable name, setup commands, and requested Slurm resources without assuming a
cluster configuration.

In [ ]:
from pathlib import Path

from IPython.display import display
from scgsim.palace import EigenmodeSim, resolve_palace_result
from scgsim.sgb import build_component_stack
from scgsim.visualization import inspect_palace_geometry

from qpdk import LAYER, LAYER_STACK, PDK
from qpdk.cells.transmon import double_pad_transmon
from qpdk.tech import material_properties

PDK.activate()

WORKFLOW_ACTION = "prepare"  # "prepare" or "analyze-returned"
RUN_ID = "transmon-surface-epr-001"
RUN_ROOT = Path(".artifacts") / "palace_transmon_surface_epr"
RUN_DIR = RUN_ROOT / RUN_ID

if WORKFLOW_ACTION not in {"prepare", "analyze-returned"}:
    raise ValueError("WORKFLOW_ACTION must be 'prepare' or 'analyze-returned'.")

## Build Component Coupon

QPDK owns the layer stack, material records, conductor identities, nets, and
selector points. SCGSim compiles those authored facts; this notebook does not
reconstruct them from geometry.

In [ ]:
PAD_SIZE_UM = (250.0, 400.0)
PAD_GAP_UM = 15.0
JUNCTION_LUMPED_PORT_WIDTH_UM = 1.0
COUPON_PADDING_UM = 100.0

component = double_pad_transmon(
    pad_size=PAD_SIZE_UM,
    pad_gap=PAD_GAP_UM,
    with_junction_lumped_port=True,
    junction_lumped_port_width=JUNCTION_LUMPED_PORT_WIDTH_UM,
    layer_simulation=LAYER.SIM_BOUNDARY,
)
stack = build_component_stack(
    component=component,
    layer_stack=LAYER_STACK,
    material_records=material_properties,
    coupon_padding_um=COUPON_PADDING_UM,
)
component.plot()

## Configure EPR / Problem

The three 2 nm interface layers below are a caller-selected loss model based on
representative MA, MS, and SA parameters reported by
:cite:p:`woodsDeterminingInterfaceDielectric2019`. They are simulation inputs for
this example, not QPDK process truth. The junction inductance is likewise an
explicit caller model rather than a fabricated-junction parameter supplied by QPDK.

In [ ]:
SURFACE_EPR_SPECS = {
    "MA": {
        "thickness": 0.002,
        "permittivity": 10.0,
        "loss_tangent": 0.0033,
    },
    "MS": {
        "thickness": 0.002,
        "permittivity": 11.4,
        "loss_tangent": 0.00048,
    },
    "SA": {
        "thickness": 0.002,
        "permittivity": 4.0,
        "loss_tangent": 0.0017,
    },
}
JUNCTION_INDUCTANCE_H = 7e-9
NUM_MODES = 2

if WORKFLOW_ACTION == "prepare":
    if RUN_DIR.exists():
        raise FileExistsError(
            f"{RUN_DIR} already exists; choose a fresh RUN_ID for preparation."
        )
    sim = EigenmodeSim()
    sim.set_geometry(component)
    sim.set_stack(stack)
    sim.set_output_dir(RUN_DIR)
    sim.set_surface_epr(
        representation="A",
        specs=SURFACE_EPR_SPECS,
        route_a_thin_film="metal_gap_equivalent",
    )
    sim.add_port(
        "junction_lumped",
        layer="M1",
        layout_sheet=True,
        inductance=JUNCTION_INDUCTANCE_H,
    )
    sim.set_eigenmode(num_modes=NUM_MODES)

## Build Mesh

These Route A sizes are visible starting values in micrometres. Review the mesh
and quality evidence before treating a returned solve as scientific evidence.

In [ ]:
REFINED_MESH_SIZE_UM = 2.0
MAX_MESH_SIZE_UM = 40.0

if WORKFLOW_ACTION == "prepare":
    sim.set_mesh(
        refined_mesh_size=REFINED_MESH_SIZE_UM,
        max_mesh_size=MAX_MESH_SIZE_UM,
    )
    mesh_path = sim.mesh()
    display(mesh_path)

## Generate Config

AMR is explicitly disabled for this handoff (`MaxIts = 0`), and the generated
Palace configuration explicitly keeps nonconformal refinement disabled.

In [ ]:
FEM_ORDER = 1
LINEAR_TOLERANCE = 1e-6
LINEAR_MAX_ITERATIONS = 400
AMR_MAX_PASSES = 0
AMR_NONCONFORMAL = False
AMR_TOLERANCE = 1e-2

if WORKFLOW_ACTION == "prepare":
    sim.set_numerical(
        order=FEM_ORDER,
        tolerance=LINEAR_TOLERANCE,
        max_iterations=LINEAR_MAX_ITERATIONS,
        solver_type="Default",
        preconditioner="Default",
        device="CPU",
        amr_max_passes=AMR_MAX_PASSES,
        amr_nonconformal=AMR_NONCONFORMAL,
        amr_tolerance=AMR_TOLERANCE,
        output_paraview=True,
    )
    config_path = sim.write_config()
    display(config_path)

## Prepare Handoff

This stage creates the portable archive and `run_palace.sbatch`; it does not run
`sbatch`. Adjust the executable, setup commands, and resource request to the target
cluster before preparing a new handoff.

In [ ]:
PALACE_EXECUTABLE = "palace"
SLURM_SETUP_COMMANDS = ()
SLURM_RESOURCES = {
    "nodes": 1,
    "ntasks": 4,
    "cpus_per_task": 1,
    "time": "01:00:00",
    "mem": "16G",
    "job_name": "qpdk-transmon-epr",
}

if WORKFLOW_ACTION == "prepare":
    handoff = sim.prepare_handoff(
        profile="slurm-single-node",
        executable=PALACE_EXECUTABLE,
        resources=SLURM_RESOURCES,
        setup_commands=SLURM_SETUP_COMMANDS,
    )
    display({
        "handoff_id": handoff.handoff_id,
        "archive": handoff.archive_path,
        "script": handoff.script_path,
    })

## Analyze Returned Run

In preparation mode, inspect the generated mesh's structured Surface-EPR
assignments before sending the archive to the cluster. After the completed package
is returned, change `WORKFLOW_ACTION` to `"analyze-returned"`, keep the same
`RUN_ID`, and paste the exact handoff ID displayed above into the stage-local control
below. Resolution verifies the returned receipt against that independent expected
identity before any report is shown.

The report order is run identity and numerical evidence, simulation cost, then
physics quantities. A returned package without bound Surface-EPR snapshots fails
rather than presenting an empty report as success.

In [ ]:
EXPECTED_HANDOFF_ID = None
REPORT_THEME = "light"
SURFACE_RANKING_LIMIT = 20

preview = inspect_palace_geometry(RUN_DIR)
display(preview.show_surface_epr())

if WORKFLOW_ACTION == "analyze-returned":
    if not EXPECTED_HANDOFF_ID:
        raise ValueError(
            "Set EXPECTED_HANDOFF_ID to the exact ID recorded during preparation."
        )
    result = resolve_palace_result(
        RUN_DIR,
        expected_handoff_id=EXPECTED_HANDOFF_ID,
    )
    trust_report = result.show_run_trustworthiness(theme=REPORT_THEME)
    benchmark_report = result.show_simulation_benchmark()
    physics_report = result.show_physics_quantities(
        theme=REPORT_THEME,
        ranking_limit=SURFACE_RANKING_LIMIT,
    )
    if not physics_report.snapshots:
        raise RuntimeError(
            "Returned run has no Surface-EPR snapshots bound to structured semantics."
        )
    display(trust_report)
    display(benchmark_report)
    display(physics_report)